# Wholesale Customers – Decision Support System
## Customer Prioritization Using a Greedy Heuristic

This notebook implements the customer-prioritization Decision Support System (DSS) described in Sections 1–6 of the project.

The model uses the **Wholesale Customers** dataset. For each customer, it calculates total annual spending across six product categories, ranks customers from highest to lowest spending, and selects the top customers under different sales-visit capacity scenarios (**K = 5, 10, and 20**).

### Decision objective
Select up to **K customers** whose combined historical annual spending is as high as possible.

### Decision rule
Because the model assumes that every customer visit requires the same amount of resources, it ranks customers by total annual spending and selects the top K. Under this equal-visit-cost assumption, this rule maximizes the historical spending represented by the selected customers for each fixed K.

### Important interpretation
The spending values are reported in the dataset's monetary units; no specific currency is assumed. The results describe **historical spending represented by selected customers**. They are not a forecast of future revenue or a guarantee of future sales.

### Notebook structure
1. Import libraries and load the dataset.
2. Validate and prepare the customer data.
3. Define and demonstrate the Greedy selection function.
4. Compare visit-capacity scenarios.
5. Present the detailed recommendation for the main scenario (K = 10).
6. Summarize the results and model limitations.

### 1 Import Libraries and Load the Dataset

The notebook uses `pandas` to read, process, rank, and display the customer records.

**Dataset location:**

```text
wholesale-customer-intelligence-dss/
└── data/
    └── Wholesale customers data.csv
```

Place the CSV file inside the `data` folder using the filename shown above. The code checks the `data` folder first and then the project folder. If the CSV is not found, it displays a clear message explaining where to place it.


In [ ]:
import pandas as pd
from pathlib import Path

# Read the dataset
DATA_FILENAME = "Wholesale customers data.csv"

# Check the data folder first, then the notebook's project folder
possible_locations = [
    Path("data") / DATA_FILENAME,
    Path(DATA_FILENAME)
]

data_file = next((file for file in possible_locations if file.is_file()), None)

if data_file is None:
    expected_locations = "\n".join(f"- {file}" for file in possible_locations)
    raise FileNotFoundError(
        f"Dataset not found. Place '{DATA_FILENAME}' in one of these locations:\n"
        f"{expected_locations}"
    )

customers = pd.read_csv(data_file)

print(f"Dataset loaded successfully from: {data_file.as_posix()}")
print(f"Dataset shape: {customers.shape}")
display(customers.head())

print("\nMissing values by column:")
display(customers.isnull().sum().to_frame("Missing Values"))

**Explanation:** The code imports `pandas` for data manipulation and `Path` for checking the dataset's location. It searches for the CSV in the repository's `data` folder and, as a fallback, in the same folder as the notebook. This makes the notebook portable across computers and avoids a personal absolute file path.

After loading the CSV into the `customers` DataFrame, the code displays the dataset dimensions, the first five records, and the missing-value count for each column. The dimensions and missing-value results are generated from the file at runtime rather than written as fixed values in the explanation.

### 2 Validate and Prepare the Customer Data

The model uses six annual spending attributes:

- `Fresh`
- `Milk`
- `Grocery`
- `Frozen`
- `Detergents_Paper`
- `Delicassen`

Before calculating customer totals, the code checks that all required columns exist and that the spending values are numeric and complete. It then creates a generated customer identifier because the source dataset does not provide a unique customer ID.

In [ ]:
spending_columns = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen"
]

required_columns = ["Channel", "Region"] + spending_columns
missing_columns = [column for column in required_columns if column not in customers.columns]

if missing_columns:
    raise ValueError(
        "The dataset is missing the following required columns: "
        + ", ".join(missing_columns)
    )

# Convert spending fields to numeric values; invalid entries become missing values
customers[spending_columns] = customers[spending_columns].apply(
    pd.to_numeric, errors="coerce"
)

if customers[spending_columns].isnull().any().any():
    raise ValueError(
        "One or more spending values are missing or non-numeric. "
        "Please check the source dataset before running the model."
    )

if (customers[spending_columns] < 0).any().any():
    raise ValueError("Spending values must be non-negative for this model.")

# Generate an identifier from each record's original row position
customers = customers.copy()
customers["Customer_ID"] = customers.index + 1

# Calculate each customer's total annual spending across the six categories
customers["Total_Spending"] = customers[spending_columns].sum(axis=1)

# Rank customers by total spending; use Customer_ID to make ties reproducible
ranked_customers = customers.sort_values(
    by=["Total_Spending", "Customer_ID"],
    ascending=[False, True],
    kind="mergesort"
).reset_index(drop=True)

total_company_spending = customers["Total_Spending"].sum()

print(f"Number of customers: {len(customers):,}")
print(f"Total historical spending: {total_company_spending:,.0f} monetary units")

display(
    ranked_customers[
        ["Customer_ID", "Channel", "Region"] + spending_columns + ["Total_Spending"]
    ].head(10)
)

**Explanation:** The code first verifies that the dataset contains the two descriptive fields (`Channel` and `Region`) and all six spending fields required by the model. Spending columns are converted to numeric values, and the code stops with an informative error if any spending value is missing, non-numeric, or negative. This prevents incomplete or invalid values from silently affecting the ranking.

Next, `Customer_ID` is generated from the original row position because the dataset does not contain a unique customer identifier. `Total_Spending` is calculated for each customer by adding the six category values in that customer's row. The customers are then sorted in descending order of total spending. If two customers have the same total, the customer with the lower generated ID is listed first. Finally, the code calculates the total historical spending across all customers and displays the ten highest-ranked records.

### 3 Define the Greedy Heuristic Algorithm

The Greedy rule makes the selection one customer at a time, starting with the customer who has the highest `Total_Spending`. It continues down the ranked list until the visit capacity is reached.

For a fixed visit capacity K, the model therefore selects the first K customers from the ranked dataset. The function also checks that K is a valid positive integer and does not exceed the number of available customers.

In [ ]:
def greedy_customer_selection(ranked_data, visit_capacity):
    """Select the highest-spending customers up to the visit capacity."""

    if not isinstance(visit_capacity, int) or isinstance(visit_capacity, bool):
        raise TypeError("Visit capacity must be an integer.")

    if visit_capacity < 1:
        raise ValueError("Visit capacity must be at least 1.")

    if visit_capacity > len(ranked_data):
        raise ValueError(
            "Visit capacity cannot exceed the number of available customers."
        )

    selected_customers = ranked_data.head(visit_capacity).copy()
    covered_spending = selected_customers["Total_Spending"].sum()

    return selected_customers, covered_spending


# Demonstrate the selection rule with a capacity of five visits
sample_capacity = 5
sample_selected, sample_covered = greedy_customer_selection(
    ranked_customers, sample_capacity
)

display(sample_selected[["Customer_ID", "Total_Spending"]])

print(f"Visit capacity: {sample_capacity}")
print(f"Customers selected: {len(sample_selected)}")
print(f"Historical spending covered: {sample_covered:,.0f} monetary units")

**Explanation:** The `greedy_customer_selection()` function receives the already-ranked customer table and the available visit capacity. It validates the capacity, selects the first K records using `head(K)`, and calculates the sum of `Total_Spending` for the selected customers.

The example runs the function with K = 5 to demonstrate the selection process. Since the customers are already ranked from highest to lowest spending, selecting the first five gives the highest possible combined historical spending among all groups of five customers, provided that each visit has the same resource cost. This optimality statement applies to the model's stated objective and assumption; it does not mean the selection is necessarily optimal for every wider business objective.

### 4 Compare Different Visit-Capacity Scenarios

The DSS is evaluated under three visit-capacity scenarios: **K = 5, K = 10, and K = 20**. The ranked customer list remains the same in all scenarios; only the number of customers that can be selected changes.

For each scenario, the model calculates:
- the visit capacity;
- the number of customers selected;
- the combined historical spending represented by those customers; and
- the selected spending as a percentage of total dataset spending.

In [ ]:
visit_capacities = [5, 10, 20]
scenario_rows = []

for k in visit_capacities:
    selected, covered = greedy_customer_selection(ranked_customers, k)

    scenario_rows.append({
        "Visit Capacity (K)": k,
        "Customers Selected": len(selected),
        "Covered Spending": covered,
        "Share of Total Spending (%)": (
            covered / total_company_spending * 100
            if total_company_spending else 0
        )
    })

scenario_results_df = pd.DataFrame(scenario_rows)

display(
    scenario_results_df.style.format({
        "Covered Spending": "{:,.0f}",
        "Share of Total Spending (%)": "{:.2f}%"
    })
)

**Explanation:** The loop runs the same Greedy selection function for each capacity in `[5, 10, 20]`. For every K, it records how many customers were selected, the sum of their historical spending, and that sum as a percentage of total spending in the dataset.

The percentage is calculated as:

\[
\text{Share of Total Spending} =
\frac{\text{Covered Spending}}{\text{Total Dataset Spending}} \times 100
\]

The resulting table allows the sales manager to compare the historical spending represented under different visit limits. A larger K will include more customers and, because the spending values are non-negative, will not reduce the total spending covered. These percentages are descriptive measures of historical data, not predicted revenue.

### 5 Detailed DSS Recommendation for the Main Scenario (K = 10)

The main scenario uses a visit capacity of **K = 10**. This section displays the ten selected customers and their customer attributes and category-level spending so the sales manager can review the recommendation in detail.

In [ ]:
main_capacity = 10
main_selected, main_covered = greedy_customer_selection(
    ranked_customers, main_capacity
)

main_recommendation = main_selected[
    ["Customer_ID", "Channel", "Region"] + spending_columns + ["Total_Spending"]
].copy()

main_share = (
    main_covered / total_company_spending
    if total_company_spending else 0
)

print(f"Main scenario: {main_capacity} visits")
print(f"Customers selected: {len(main_recommendation)}")
print(f"Historical spending covered: {main_covered:,.0f} monetary units")
print(f"Share of total dataset spending: {main_share:.2%}")

display(
    main_recommendation.style.format({
        column: "{:,.0f}" for column in spending_columns + ["Total_Spending"]
    })
)

**Explanation:** The code applies the Greedy function to the main scenario with K = 10. It then creates a detailed table containing each selected customer's generated ID, channel, region, spending in each of the six product categories, and total spending.

The printed summary reports the combined historical spending of the ten selected customers and its share of total dataset spending. This detailed output helps the sales manager review which accounts were prioritized and understand the historical spending information behind the recommendation. K = 10 is used as the main detailed example; the K = 5 and K = 20 scenarios remain available in the comparison table.

### 6 Final DSS Results

The final results table combines the scenario comparison with a short decision statement for each visit capacity. It provides a concise view of how the visit limit affects the number of selected customers and the historical spending represented by them.

In [ ]:
final_results = scenario_results_df.copy()

final_results["DSS Decision"] = final_results["Visit Capacity (K)"].apply(
    lambda k: f"Prioritize the top {k} customers"
)

display(
    final_results.style.format({
        "Covered Spending": "{:,.0f}",
        "Share of Total Spending (%)": "{:.2f}%"
    })
)

print(
    "Decision rule: rank customers by Total_Spending in descending order "
    "and select the top K customers."
)

**Explanation:** The code copies the scenario comparison table and adds a decision statement that identifies the number of customers to prioritize under each capacity. The table summarizes the model's outputs in a format that can support the sales manager's comparison of alternative visit limits.

The recommended customer list is determined by the same transparent rule in every scenario: rank customers by total historical spending and select the first K records. The manager can then choose the visit capacity that best matches the sales team's available resources.

### 7 Interpretation and Evaluation

The scenario table should be used to compare the historical spending represented by the selected customers under each visit capacity. The model provides a simple and transparent way to prioritize customer visits when the number of available visits is limited.

**Model strengths**
- It converts six spending fields into one comparable customer-level measure.
- It produces a clear and reproducible priority ranking.
- It supports what-if comparison across different visit capacities.
- Sorting the customer records requires approximately **O(n log n)** time; selecting the first K records after sorting is efficient.

**Model assumptions and limitations**
- Every customer visit is assumed to require equal resources.
- Total historical spending is the only criterion used to rank customers.
- The model does not consider profit margins, visit costs, travel distance, customer growth potential, or future purchasing behavior.
- A generated `Customer_ID` is only a row-based identifier for this dataset; it is not an original customer account number.
- High historical spending does not guarantee high future sales.

**Overall conclusion:** The Greedy Heuristic transforms the wholesale customer data into a ranked list and a set of visit-capacity recommendations. It supports the sales manager's decision by showing which customers account for the highest historical spending, while the final business decision should also consider operational and strategic factors that are outside the current model.